# 📖 Notebook 2: Privacy Impact Assessment (PIA)

A **Privacy Impact Assessment** is a structured process to evaluate the privacy risks of a new feature **before** it launches. At Microsoft, Google, and other large tech companies, no feature that handles personal data can ship without a completed PIA.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to build a PIA workflow with structured risk evaluation
- How to score privacy risks using a consistent framework
- How to document data flows (what data goes where)
- How to track PIA status through the review lifecycle
- Why PIAs exist and what happens when you skip them

## 🛠️ Setup

Start the infrastructure first:

```bash
cd enterprise-patterns/privacy-review
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
from datetime import datetime, timedelta

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "privacy_review",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 Why Do PIAs Exist?

Imagine you're building a new feature: **"Personalized Product Recommendations"**. Sounds harmless, right?

But think about what it needs:
- User's **browsing history** (every page they visited)
- User's **purchase history** (what they bought)
- User's **location** (to recommend local products)
- Maybe even **demographic data** (age, gender) for better targeting

Now ask:
- What if this data is **shared with a third-party ML vendor**?
- What if the ML model **discriminates** against certain demographics?
- What if the browsing history **reveals sensitive health or political interests**?
- What if the data is **transferred to servers in another country** with weaker privacy laws?

A PIA forces you to answer these questions **before** writing any code. It's much cheaper to change a design on paper than to refactor a shipped feature.

## 📋 Step 1: Define the PIA Framework

A PIA evaluates risk across several dimensions. Let's define our scoring framework.

In [ ]:
# The PIA risk framework — each dimension contributes to the overall risk score

RISK_DIMENSIONS = {
    "data_sensitivity": {
        "description": "How sensitive is the data being collected?",
        "scoring": {
            1: "Only public data (product names, public pages)",
            2: "Internal data (usage counts, aggregated metrics)",
            3: "Confidential PII (names, emails, phone numbers)",
            4: "Restricted PII (SSN, financial, health data)",
            5: "Special categories (biometric, genetic, children's data)"
        }
    },
    "data_volume": {
        "description": "How many people are affected?",
        "scoring": {
            1: "< 100 users (internal testing)",
            2: "100–10,000 users (limited release)",
            3: "10,000–1M users (regional launch)",
            4: "1M–100M users (major product)",
            5: "> 100M users (global platform)"
        }
    },
    "third_party_sharing": {
        "description": "Is data shared outside the company?",
        "scoring": {
            1: "No sharing — data stays in our systems",
            2: "Shared with processors under contract (cloud hosting)",
            3: "Shared with partners for joint features",
            4: "Shared with advertisers or data brokers",
            5: "Made publicly available or sold"
        }
    },
    "cross_border_transfer": {
        "description": "Does data cross country borders?",
        "scoring": {
            1: "No — stays in one country",
            2: "Within same legal region (EU-to-EU, US states)",
            3: "Between adequate countries (EU-to-Canada)",
            4: "To countries without adequacy (EU-to-US without framework)",
            5: "To countries with poor privacy protections"
        }
    },
    "automated_decisions": {
        "description": "Are automated decisions made about people?",
        "scoring": {
            1: "No automated decisions",
            2: "Recommendations (user can ignore)",
            3: "Content filtering or ranking",
            4: "Eligibility decisions (credit, insurance, hiring)",
            5: "Autonomous actions with real-world impact"
        }
    },
    "retention_period": {
        "description": "How long is data kept?",
        "scoring": {
            1: "Not stored — processed and discarded",
            2: "< 30 days (session data, temporary)",
            3: "30 days – 1 year (operational use)",
            4: "1–7 years (legal requirements)",
            5: "Indefinite or no defined retention"
        }
    }
}

# Print the framework so users can reference it
print("📋 PIA Risk Scoring Framework")
print("=" * 70)
for dim_name, dim in RISK_DIMENSIONS.items():
    print(f"\n📊 {dim_name.replace('_', ' ').title()}")
    print(f"   {dim['description']}")
    for score, desc in dim["scoring"].items():
        print(f"   {score} — {desc}")

## 🏗️ Step 2: Build the PIA Workflow Engine

Now let's build the actual PIA system. It will:
1. Accept a feature description and data flow
2. Score each risk dimension
3. Calculate an overall risk level
4. Determine what approvals are needed
5. Store the PIA for tracking

In [ ]:
class PrivacyImpactAssessment:
    """A complete PIA workflow engine."""

    # Overall risk thresholds
    RISK_THRESHOLDS = {
        (1.0, 1.5): {"level": "negligible", "approval": "self-approve",   "color": "🟢"},
        (1.5, 2.5): {"level": "low",        "approval": "team lead",      "color": "🔵"},
        (2.5, 3.5): {"level": "medium",     "approval": "privacy team",   "color": "🟡"},
        (3.5, 4.5): {"level": "high",       "approval": "privacy + legal","color": "🟠"},
        (4.5, 5.1): {"level": "critical",   "approval": "CISO + DPO",    "color": "🔴"},
    }

    def __init__(self, feature_name, team, assessor):
        self.feature_name = feature_name
        self.team = team
        self.assessor = assessor
        self.description = ""
        self.data_flows = []
        self.scores = {}
        self.mitigations = []
        self.status = "draft"
        self.created_at = datetime.now()

    def set_description(self, description):
        """Describe what the feature does."""
        self.description = description

    def add_data_flow(self, source, destination, data_types, purpose):
        """Document a data flow: where data comes from and where it goes."""
        self.data_flows.append({
            "source": source,
            "destination": destination,
            "data_types": data_types,
            "purpose": purpose
        })

    def score_dimension(self, dimension, score, justification=""):
        """Score a risk dimension (1-5)."""
        if dimension not in RISK_DIMENSIONS:
            raise ValueError(f"Unknown dimension: {dimension}")
        if score < 1 or score > 5:
            raise ValueError("Score must be 1-5")
        self.scores[dimension] = {
            "score": score,
            "justification": justification
        }

    def add_mitigation(self, risk, mitigation, reduces_score_by=0):
        """Add a mitigation measure for an identified risk."""
        self.mitigations.append({
            "risk": risk,
            "mitigation": mitigation,
            "score_reduction": reduces_score_by
        })

    def calculate_risk(self):
        """Calculate overall risk score and determine approval requirements."""
        if not self.scores:
            return None

        # Average of all scored dimensions
        avg_score = sum(s["score"] for s in self.scores.values()) / len(self.scores)

        # Apply mitigation reductions
        total_reduction = sum(m["score_reduction"] for m in self.mitigations)
        adjusted_score = max(1.0, avg_score - (total_reduction / max(len(self.scores), 1)))

        # Determine risk level
        for (low, high), info in self.RISK_THRESHOLDS.items():
            if low <= adjusted_score < high:
                return {
                    "raw_score": round(avg_score, 2),
                    "adjusted_score": round(adjusted_score, 2),
                    "level": info["level"],
                    "approval_required": info["approval"],
                    "color": info["color"]
                }

        return {"raw_score": round(avg_score, 2), "adjusted_score": round(adjusted_score, 2),
                "level": "unknown", "approval_required": "review needed", "color": "⚪"}

    def generate_report(self):
        """Generate a human-readable PIA report."""
        risk = self.calculate_risk()

        report = []
        report.append("=" * 70)
        report.append("PRIVACY IMPACT ASSESSMENT REPORT")
        report.append("=" * 70)
        report.append(f"Feature:    {self.feature_name}")
        report.append(f"Team:       {self.team}")
        report.append(f"Assessor:   {self.assessor}")
        report.append(f"Date:       {self.created_at.strftime('%Y-%m-%d')}")
        report.append(f"Status:     {self.status.upper()}")
        report.append("")
        report.append(f"Description: {self.description}")

        # Data flows
        report.append("\n" + "-" * 70)
        report.append("DATA FLOWS")
        report.append("-" * 70)
        for i, flow in enumerate(self.data_flows, 1):
            report.append(f"  Flow {i}: {flow['source']} → {flow['destination']}")
            report.append(f"    Data:    {', '.join(flow['data_types'])}")
            report.append(f"    Purpose: {flow['purpose']}")

        # Risk scores
        report.append("\n" + "-" * 70)
        report.append("RISK SCORES")
        report.append("-" * 70)
        for dim, info in self.scores.items():
            bar = "█" * info["score"] + "░" * (5 - info["score"])
            report.append(f"  {dim.replace('_', ' ').title():<28} [{bar}] {info['score']}/5")
            if info["justification"]:
                report.append(f"    ↳ {info['justification']}")

        # Mitigations
        if self.mitigations:
            report.append("\n" + "-" * 70)
            report.append("MITIGATIONS")
            report.append("-" * 70)
            for m in self.mitigations:
                report.append(f"  🛡️ Risk: {m['risk']}")
                report.append(f"     Fix:  {m['mitigation']}")
                if m["score_reduction"]:
                    report.append(f"     Reduces risk by: {m['score_reduction']}")

        # Overall risk
        if risk:
            report.append("\n" + "=" * 70)
            report.append("OVERALL RISK ASSESSMENT")
            report.append("=" * 70)
            report.append(f"  {risk['color']} Risk Level:      {risk['level'].upper()}")
            report.append(f"     Raw Score:       {risk['raw_score']}/5.0")
            report.append(f"     Adjusted Score:  {risk['adjusted_score']}/5.0 (after mitigations)")
            report.append(f"     Approval Needed: {risk['approval_required']}")

        return "\n".join(report)

print("✅ PIA engine loaded")

## 📝 Step 3: Create a PIA for a Real Feature

Let's walk through a complete PIA for a realistic feature: **"Personalized Product Recommendations"**.

This is the type of feature that would trigger a PIA at Microsoft, Google, or Amazon because it:
- Collects behavioral data (browsing, purchases)
- Uses automated decision-making (ML model)
- May involve a third-party ML service
- Processes data from millions of users

In [ ]:
# Create the PIA
pia = PrivacyImpactAssessment(
    feature_name="Personalized Product Recommendations",
    team="Product Engineering",
    assessor="privacy-lab-student"
)

# Describe the feature
pia.set_description(
    "ML-powered recommendation engine that suggests products based on "
    "browsing history, purchase history, and user demographics. "
    "Recommendations appear on the homepage and product pages."
)

# Document data flows — where does data move?
pia.add_data_flow(
    source="User Browser",
    destination="Activity Log (PostgreSQL)",
    data_types=["page views", "click events", "search queries", "IP address"],
    purpose="Track user behavior to train recommendation model"
)

pia.add_data_flow(
    source="Activity Log (PostgreSQL)",
    destination="ML Training Pipeline (internal)",
    data_types=["anonymized user IDs", "product interactions", "timestamps"],
    purpose="Train recommendation model on aggregated behavior patterns"
)

pia.add_data_flow(
    source="ML Model",
    destination="User Browser (via API)",
    data_types=["recommended product IDs", "relevance scores"],
    purpose="Display personalized recommendations to the user"
)

pia.add_data_flow(
    source="User Profile (PostgreSQL)",
    destination="ML Training Pipeline",
    data_types=["age range", "city", "signup date"],
    purpose="Demographic features for model training (no names or emails)"
)

print("✅ Feature description and data flows documented")
print(f"   {len(pia.data_flows)} data flows registered")

In [ ]:
# Score each risk dimension with justification

pia.score_dimension(
    "data_sensitivity", 3,
    "Collects browsing behavior and demographics — Confidential PII"
)

pia.score_dimension(
    "data_volume", 4,
    "Feature targets all users — estimated 5M active users"
)

pia.score_dimension(
    "third_party_sharing", 1,
    "ML model runs internally — no data leaves our infrastructure"
)

pia.score_dimension(
    "cross_border_transfer", 2,
    "ML training runs in US-West, users are US-only for now"
)

pia.score_dimension(
    "automated_decisions", 2,
    "Recommendations only — user can ignore them, no access is restricted"
)

pia.score_dimension(
    "retention_period", 3,
    "Activity logs kept 90 days, model retrained monthly"
)

# Add mitigations to reduce risk
pia.add_mitigation(
    risk="Browsing data reveals sensitive interests (health, politics)",
    mitigation="Exclude health and political categories from recommendation features",
    reduces_score_by=0.5
)

pia.add_mitigation(
    risk="Model could discriminate by demographics",
    mitigation="Run fairness audit on model outputs before deployment",
    reduces_score_by=0.3
)

pia.add_mitigation(
    risk="Users cannot control what data is used",
    mitigation="Add opt-out toggle in user privacy settings",
    reduces_score_by=0.2
)

# Generate and print the full report
print(pia.generate_report())

## 💾 Step 4: Save the PIA to the Database

PIAs need to be stored for compliance — regulators may ask to see them years later. Let's save our PIA to PostgreSQL and cache the status in Redis.

In [ ]:
def save_pia(pia):
    """Save a PIA to the database."""
    risk = pia.calculate_risk()
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO privacy_impact_assessments
            (feature_name, description, team, assessor,
             data_types_collected, purpose, third_party_sharing,
             cross_border_transfer, automated_decision_making,
             risk_score, status, expires_at)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        RETURNING id
    """, (
        pia.feature_name,
        pia.description,
        pia.team,
        pia.assessor,
        json.dumps([f["data_types"] for f in pia.data_flows]),
        json.dumps([f["purpose"] for f in pia.data_flows]),
        pia.scores.get("third_party_sharing", {}).get("score", 1) > 2,
        pia.scores.get("cross_border_transfer", {}).get("score", 1) > 2,
        pia.scores.get("automated_decisions", {}).get("score", 1) > 2,
        risk["adjusted_score"] if risk else None,
        "in_review",
        datetime.now() + timedelta(days=365)  # PIAs expire after 1 year
    ))

    pia_id = cursor.fetchone()[0]
    conn.commit()
    conn.close()

    # Cache in Redis for quick status lookups
    r = get_redis_client()
    r.hset(f"pia:{pia.feature_name}", mapping={
        "id": str(pia_id),
        "status": "in_review",
        "risk_level": risk["level"] if risk else "unknown",
        "risk_score": str(risk["adjusted_score"]) if risk else "0",
        "approval_required": risk["approval_required"] if risk else "unknown",
        "expires_at": (datetime.now() + timedelta(days=365)).isoformat()
    })

    return pia_id

pia_id = save_pia(pia)
print(f"✅ PIA saved with ID: {pia_id}")
print(f"   Status: in_review")
print(f"   Expires: {(datetime.now() + timedelta(days=365)).strftime('%Y-%m-%d')}")

# Show it in Redis
r = get_redis_client()
cached = r.hgetall(f"pia:{pia.feature_name}")
print(f"\n🔍 Redis cache: {json.dumps(cached, indent=2)}")

## 📊 Step 5: PIA Review Dashboard

In a real company, the privacy team needs a dashboard to see all pending PIAs, their risk levels, and which ones need attention. Let's build a simple one.

In [ ]:
# First, let's create a few more PIAs so the dashboard has data to show

sample_features = [
    {
        "name": "Email Newsletter Signup",
        "team": "Marketing",
        "description": "Collect email and name for weekly newsletter",
        "scores": {"data_sensitivity": 3, "data_volume": 3, "third_party_sharing": 3,
                   "cross_border_transfer": 1, "automated_decisions": 1, "retention_period": 4}
    },
    {
        "name": "Customer Support Chat",
        "team": "Support Engineering",
        "description": "Live chat with support agents, transcripts stored",
        "scores": {"data_sensitivity": 4, "data_volume": 3, "third_party_sharing": 1,
                   "cross_border_transfer": 1, "automated_decisions": 1, "retention_period": 3}
    },
    {
        "name": "AI Credit Scoring",
        "team": "FinTech",
        "description": "ML model that scores creditworthiness using financial data",
        "scores": {"data_sensitivity": 5, "data_volume": 4, "third_party_sharing": 2,
                   "cross_border_transfer": 3, "automated_decisions": 5, "retention_period": 4}
    },
    {
        "name": "Public Product Reviews",
        "team": "Product Engineering",
        "description": "Users post public reviews with display name and rating",
        "scores": {"data_sensitivity": 1, "data_volume": 3, "third_party_sharing": 1,
                   "cross_border_transfer": 1, "automated_decisions": 1, "retention_period": 5}
    },
]

for feature in sample_features:
    p = PrivacyImpactAssessment(feature["name"], feature["team"], "auto-assessment")
    p.set_description(feature["description"])
    for dim, score in feature["scores"].items():
        p.score_dimension(dim, score)
    save_pia(p)

print(f"✅ Created {len(sample_features)} additional PIAs")

In [ ]:
def display_pia_dashboard():
    """Show all PIAs in a dashboard format."""
    conn = get_db_connection()
    cursor = conn.cursor(psycopg2.extras.RealDictCursor)

    cursor.execute("""
        SELECT id, feature_name, team, risk_score, status,
               third_party_sharing, cross_border_transfer,
               automated_decision_making, created_at, expires_at
        FROM privacy_impact_assessments
        ORDER BY risk_score DESC NULLS LAST
    """)

    pias = cursor.fetchall()
    conn.close()

    print("📊 Privacy Impact Assessment Dashboard")
    print("=" * 90)
    print(f"  {'ID':<4} {'Feature':<35} {'Team':<20} {'Risk':>5} {'Status':<12} {'Flags'}")
    print("-" * 90)

    for p in pias:
        # Risk color
        score = float(p["risk_score"]) if p["risk_score"] else 0
        if score >= 4.0:
            icon = "🔴"
        elif score >= 3.0:
            icon = "🟠"
        elif score >= 2.0:
            icon = "🟡"
        else:
            icon = "🟢"

        # Risk flags
        flags = []
        if p["third_party_sharing"]:
            flags.append("3rd-party")
        if p["cross_border_transfer"]:
            flags.append("cross-border")
        if p["automated_decision_making"]:
            flags.append("auto-decision")

        print(f"  {p['id']:<4} {p['feature_name'][:34]:<35} {p['team'][:19]:<20} "
              f"{icon}{score:>4.1f} {p['status']:<12} {', '.join(flags) if flags else '—'}")

    # Summary
    print("\n" + "=" * 90)
    print(f"  Total PIAs: {len(pias)}")
    print(f"  ⚠️  High/Critical risk: {sum(1 for p in pias if p['risk_score'] and float(p['risk_score']) >= 3.5)}")
    print(f"  ⏰ Expiring in 30 days: {sum(1 for p in pias if p['expires_at'] and p['expires_at'] < datetime.now() + timedelta(days=30))}")

display_pia_dashboard()

## 🔄 Step 6: PIA Lifecycle Management

PIAs aren't one-and-done. They have a lifecycle:

```
draft → in_review → approved → [expires after 1 year] → renewal needed
                  ↘ rejected → revised → in_review (again)
```

Let's implement approval and expiry checking.

In [ ]:
def approve_pia(pia_id, approver):
    """Approve a PIA — changes status and records who approved it."""
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        UPDATE privacy_impact_assessments
        SET status = 'approved', approved_by = %s
        WHERE id = %s AND status = 'in_review'
        RETURNING feature_name, risk_score
    """, (approver, pia_id))

    result = cursor.fetchone()
    conn.commit()
    conn.close()

    if result:
        # Update Redis cache
        r = get_redis_client()
        r.hset(f"pia:{result[0]}", "status", "approved")
        print(f"✅ PIA #{pia_id} '{result[0]}' approved by {approver}")
    else:
        print(f"❌ PIA #{pia_id} not found or not in review")

def reject_pia(pia_id, reason):
    """Reject a PIA — feature cannot ship until issues are fixed."""
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        UPDATE privacy_impact_assessments
        SET status = 'rejected'
        WHERE id = %s AND status = 'in_review'
        RETURNING feature_name
    """, (pia_id,))

    result = cursor.fetchone()
    conn.commit()
    conn.close()

    if result:
        r = get_redis_client()
        r.hset(f"pia:{result[0]}", "status", "rejected")
        print(f"❌ PIA #{pia_id} '{result[0]}' rejected")
        print(f"   Reason: {reason}")

def check_pia_before_deploy(feature_name):
    """Check if a feature has an approved, non-expired PIA. Called in CI/CD."""
    # Try Redis cache first
    r = get_redis_client()
    cached = r.hgetall(f"pia:{feature_name}")

    if cached:
        status = cached.get("status", "unknown")
        if status != "approved":
            return False, f"PIA status is '{status}' — must be 'approved' to deploy"

        expires = cached.get("expires_at", "")
        if expires and datetime.fromisoformat(expires) < datetime.now():
            return False, "PIA has expired — renewal required"

        return True, "PIA approved and valid"

    return False, "No PIA found for this feature — create one before deploying"

# Demo: approve the low-risk PIA, reject the high-risk one
print("📝 PIA Review Decisions\n")

# Approve the recommendation engine PIA (we added mitigations)
approve_pia(1, "privacy-officer@company.com")

# Reject the AI credit scoring PIA (too risky without more mitigations)
reject_pia(4, "Automated credit decisions require human-in-the-loop override per EU AI Act")

# Check if features can deploy
print("\n🚀 Deployment Gate Checks\n")
for feature in ["Personalized Product Recommendations", "AI Credit Scoring", "Unregistered Feature"]:
    can_deploy, reason = check_pia_before_deploy(feature)
    icon = "✅" if can_deploy else "🚫"
    print(f"  {icon} {feature}")
    print(f"     {reason}")

## 🎯 Key Takeaways

1. **PIAs are mandatory gates** — no feature ships without one at companies like Microsoft
2. **Risk scoring is structured** — use consistent dimensions so PIAs are comparable
3. **Data flows are documented** — know exactly where personal data moves
4. **Mitigations reduce risk** — encryption, opt-outs, and access controls lower the score
5. **PIAs expire** — privacy landscapes change, so reassess annually
6. **Automate the gate** — check PIA status in CI/CD so features can't bypass review

### What Microsoft Does

- Every Azure service has a PIA in a central registry
- The **SDL (Security Development Lifecycle)** requires PIAs at design phase
- PIAs are linked to feature flags — rejected PIAs block deployment
- Annual PIA renewal is automated — teams get reminders 60 days before expiry
- The Chief Privacy Officer reviews all Critical-risk PIAs personally

### Next Notebook

In **Notebook 3: Anonymization Techniques**, we'll learn how to protect data using k-anonymity, l-diversity, differential privacy, and tokenization.